# Retrieval-Augmented Generation (RAG)

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/04-llm-and-transformers/04_rag_basics.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Build a simple RAG pipeline — embed documents, store in a vector index, retrieve relevant context, and feed it to an LLM for grounded answers.

**Prerequisites:** Fine-tuning (notebook 03)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy transformers sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import numpy as np

## 1. What is RAG and Why It Matters

In [ ]:
print("""
RAG (Retrieval-Augmented Generation) Architecture:

  Query → [Embed] → [Search Vector Store] → [Top-K Documents]
                                                    ↓
                                    [Combine with Prompt] → [LLM] → Answer

Why RAG?
  1. LLMs have a knowledge cutoff — RAG adds current information
  2. LLMs hallucinate — RAG grounds answers in real documents
  3. LLMs can't access private data — RAG connects them to your docs
  4. No fine-tuning needed — just update the document store
""")

## 2. Document Embedding with Sentence Transformers

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    "Python is a high-level programming language known for its readability.",
    "PyTorch is a deep learning framework developed by Meta AI.",
    "NumPy provides support for large multi-dimensional arrays and matrices.",
    "Transformers are neural network architectures based on self-attention.",
    "Kubernetes is a container orchestration platform for automating deployment.",
    "PostgreSQL is a powerful open-source relational database system.",
    "React is a JavaScript library for building user interfaces.",
    "Gradient descent is an optimization algorithm used in machine learning.",
]

embeddings = model.encode(documents)
print(f"Documents: {len(documents)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dim: {embeddings.shape[1]}")

## 3. Vector Similarity Search (Cosine Similarity)

In [ ]:
def cosine_similarity_matrix(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_norm @ b_norm.T

query = "How do neural networks learn?"
query_embedding = model.encode([query])

similarities = cosine_similarity_matrix(query_embedding, embeddings)[0]

# Rank by similarity
ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

print(f"Query: '{query}'\n")
print("Results (ranked by similarity):")
for idx, score in ranked[:5]:
    print(f"  {score:.4f} | {documents[idx]}")

## 4. Building a Simple Vector Store

In [ ]:
class SimpleVectorStore:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None

    def add(self, docs):
        self.documents.extend(docs)
        self.embeddings = self.model.encode(self.documents)

    def search(self, query, top_k=3):
        query_emb = self.model.encode([query])
        sims = cosine_similarity_matrix(query_emb, self.embeddings)[0]
        top_indices = np.argsort(sims)[::-1][:top_k]
        return [(self.documents[i], sims[i]) for i in top_indices]

store = SimpleVectorStore()
store.add(documents)

results = store.search("What tools are used for deep learning?", top_k=3)
print("Search Results:")
for doc, score in results:
    print(f"  [{score:.4f}] {doc}")

## 5. Combining Retrieval with Generation

In [ ]:
def rag_answer(question, store, top_k=3):
    """Simple RAG: retrieve context, then generate with a local model."""
    results = store.search(question, top_k=top_k)
    context = "\n".join([f"- {doc}" for doc, _ in results])

    prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:"""

    print(f"Question: {question}")
    print(f"\nRetrieved context ({top_k} docs):")
    for doc, score in results:
        print(f"  [{score:.4f}] {doc}")
    print(f"\nGenerated prompt:\n{prompt}")
    return prompt

# Demo (without calling an LLM — shows the prompt that would be sent)
rag_answer("What is PyTorch and what is it used for?", store)

## 6. Chunking Strategies for Long Documents

In [ ]:
long_document = """
Machine learning has evolved significantly over the past decade. Early approaches
focused on simple statistical methods like linear regression and decision trees.
The introduction of neural networks brought new capabilities, especially in
pattern recognition. Deep learning, powered by GPUs and large datasets, enabled
breakthroughs in image classification, speech recognition, and natural language
processing. The transformer architecture, introduced in the "Attention Is All
You Need" paper in 2017, revolutionized NLP. Models like BERT and GPT showed
that large pretrained models could be fine-tuned for specific tasks. The latest
generation of large language models demonstrates emergent abilities like
reasoning and code generation.
"""

def chunk_by_sentences(text, chunk_size=2):
    sentences = [s.strip() for s in text.strip().split('.') if s.strip()]
    chunks = []
    for i in range(0, len(sentences), chunk_size):
        chunk = '. '.join(sentences[i:i+chunk_size]) + '.'
        chunks.append(chunk)
    return chunks

def chunk_by_chars(text, chunk_size=200, overlap=50):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

print("=== Sentence-based chunks (2 sentences each) ===")
for i, chunk in enumerate(chunk_by_sentences(long_document)):
    print(f"  Chunk {i}: ({len(chunk)} chars) {chunk[:80]}...")

print(f"\n=== Character-based chunks (200 chars, 50 overlap) ===")
for i, chunk in enumerate(chunk_by_chars(long_document)):
    print(f"  Chunk {i}: ({len(chunk)} chars) {chunk[:80]}...")

## Try It Yourself

1. Build a RAG system over a set of Wikipedia paragraphs. Embed them, retrieve top-3 for a question, and generate an answer.
2. Experiment with different chunk sizes (100, 200, 500 tokens). How does chunk size affect retrieval quality?
3. Compare cosine similarity vs dot product for retrieval. When does each work better?

In [ ]:
# Your code here